# Rebuild `excitatory_neurons.csv`

The source table behind Figure 2b, 2d, 2e and S2b: one row per laminae I-II excitatory
interneuron (n = 177), with the number of synapses it receives from each sensory subtype,
its non-sensory (`N/A`) synapse count, and its subtype label.

Built by querying each ExN's input synapses from CAVE (`synapses_version3`) and classifying
every presynaptic site with SPINE, then attaching the label from
`spinalcord_neuron_clusters_2`. A synapse called sensory but with every subtype probability
<= 0.6 is counted as `low_confidence_sensory`.

The finished table ships at `../../data/sensory_input_features/excitatory_neurons.csv`, so
this only needs re-running to rebuild it. **Requires CAVE**, and the per-neuron SPINE
`.feather` outputs, which are not in this repository (see Data availability).


In [ ]:
# Libraries
import caveclient
import subprocess
import json
import numpy as np
import matplotlib.pyplot as plt

import cloudvolume
from tqdm import tqdm
import glob
import os
import json
# Setup caveclient
client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr", ) #server_address='https://global.daf-apis.com')

In [ ]:
import os
from typing import Iterable, Optional
import pandas as pd

def summarize_outputs_from_csv(
    client,
    csv_path: str = "../../data/sensory_input_features/excitatory_neurons.csv",
    *,
    table: str = "output_synapses",
    size_col: Optional[str] = "size",
    size_threshold: int = 20,
    downstream_min_synapses: int = 4,
    chunk_size: int = 512,
    progress: bool = True,
) -> pd.DataFrame:
    """
    For each neuron_id in the CSV, query its output synapses (pre_pt_root_id),
    filter out self-connections and synapses with size < size_threshold,
    then return a summary table with:
      - n_output_synapses
      - n_downstream_neurons_ge4  (partners with >= downstream_min_synapses synapses)

    Parameters
    ----------
    client : Any
        Your materialization client with .materialize.query_table(...)
    csv_path : str
        Path to CSV containing a 'neuron_id' column.
    table : str
        Synapse table name (e.g., 'output_synapses', 'axoaxonic_output_synapses', etc.).
    size_col : Optional[str]
        Name of the synapse size column to threshold on. If None, no size filtering is applied.
    size_threshold : int
        Keep synapses with size >= this value.
    downstream_min_synapses : int
        Minimum number of synapses to count a postsynaptic partner as a “downstream neuron”.
    chunk_size : int
        Batch size for pre_pt_root_id queries.
    progress : bool
        Show a tqdm progress bar if available.

    Returns
    -------
    pd.DataFrame with columns:
        neuron_id, n_output_synapses, n_downstream_neurons_ge4
    """
    # ---- Load neuron IDs
    df_ids = pd.read_csv(csv_path)
    if "neuron_id" not in df_ids.columns:
        raise KeyError(f"CSV must contain a 'neuron_id' column. Got columns: {list(df_ids.columns)}")
    # Preserve original order but ensure unique IDs for querying
    order_ids = df_ids["neuron_id"].astype(int).tolist()
    pre_ids_int = sorted({int(x) for x in order_ids})

    # ---- Query synapses in chunks
    all_chunks = []
    iterator = range(0, len(pre_ids_int), chunk_size)
    if progress:
        try:
            from tqdm import tqdm
            iterator = tqdm(iterator, desc=f"Querying {table}")
        except Exception:
            pass

    for i in iterator:
        batch = pre_ids_int[i:i+chunk_size]
        try:
            syn = client.materialize.query_table(
                table,
                filter_in_dict={"pre_pt_root_id": batch}
            )
        except Exception as e:
            print(f"⚠️ query failed for batch starting at {i}: {e}")
            continue
        if syn is None or len(syn) == 0:
            continue
        df = pd.DataFrame(syn)

        # Basic column checks
        need_cols = {"pre_pt_root_id", "post_pt_root_id"}
        if not need_cols.issubset(df.columns):
            raise KeyError(f"Table '{table}' must have columns {need_cols}. Got {list(df.columns)}")

        # Remove self-synapses
        df = df[df["pre_pt_root_id"] != df["post_pt_root_id"]]

        # Size filtering
        if size_col is not None:
            if size_col not in df.columns:
                # Try a few common alternatives before failing
                alt_cols = [size_col, "size_nm3", "size_nm^3", "synapse_size", "size_total"]
                found = None
                for c in alt_cols:
                    if c in df.columns:
                        found = c
                        break
                if found is None:
                    raise KeyError(
                        f"Size column '{size_col}' not found in table '{table}'. "
                        f"Available columns: {list(df.columns)}"
                    )
                size_col = found
            df = df[df[size_col] >= size_threshold]

        if not df.empty:
            all_chunks.append(df[["pre_pt_root_id", "post_pt_root_id"]])

    # ---- If nothing matched, return zeros for all neuron_ids
    if not all_chunks:
        out = pd.DataFrame({
            "neuron_id": order_ids,
            "n_output_synapses": [0] * len(order_ids),
            "n_downstream_neurons_ge4": [0] * len(order_ids),
        })
        return out

    syn_all = pd.concat(all_chunks, ignore_index=True)

    # Count synapses per (pre, post)
    pair_counts = (syn_all
                   .groupby(["pre_pt_root_id", "post_pt_root_id"], as_index=False)
                   .size())
    pair_counts.rename(columns={"size": "count"}, inplace=True)

    # Per-pre totals: total synapse count
    total_syn_by_pre = pair_counts.groupby("pre_pt_root_id")["count"].sum()

    # Per-pre: number of downstream partners with >= downstream_min_synapses
    dsn_ge_min = (pair_counts[pair_counts["count"] >= downstream_min_synapses]
                  .groupby("pre_pt_root_id")["post_pt_root_id"].nunique())

    # Assemble final, preserving CSV order and filling zeros
    out = pd.DataFrame({
        "neuron_id": order_ids,
    })
    out["n_output_synapses"] = out["neuron_id"].map(total_syn_by_pre).fillna(0).astype(int)
    out["n_downstream_neurons_ge4"] = out["neuron_id"].map(dsn_ge_min).fillna(0).astype(int)

    return out

# ---- Example usage (uncomment and run in your environment) ----
summary = summarize_outputs_from_csv(
     client,
     csv_path="../../data/sensory_input_features/excitatory_neurons.csv",
     table="synapses_version3",          # or "axoaxonic_output_synapses", etc.
     size_col="size",                  # adjust if your table uses a different size column
     size_threshold=20,
     downstream_min_synapses=4,
     chunk_size=512,
     progress=True,
 )
# # Save alongside the input CSV
# out_path = os.path.join(os.path.dirname("../../data/sensory_input_features/excitatory_neurons.csv"),
#                         "excitatory_neurons_output_summary.csv")
# summary.to_csv(out_path, index=False)
# print(f"Wrote {out_path}")
# summary.head()


In [ ]:
summary

In [ ]:
# Add two summary columns back into the original CSV (in place, with a backup).
import pandas as pd
from datetime import datetime
import shutil

csv_path = "../../data/sensory_input_features/excitatory_neurons.csv"

# Load original
df = pd.read_csv(csv_path)
if "neuron_id" not in df.columns:
    raise KeyError(f"'neuron_id' not in {csv_path}. Got: {list(df.columns)}")
df["neuron_id"] = df["neuron_id"].astype(int)

# ---- OPTION A: You already have a DataFrame called `summary`
# summary = <your precomputed DataFrame>

# ---- OPTION B: Or load it from disk
# summary = pd.read_csv("../../data/sensory_input_features/excitatory_neurons_output_summary.csv")

need_cols = {"neuron_id", "n_output_synapses", "n_downstream_neurons_ge4"}
missing = need_cols - set(summary.columns)
if missing:
    raise KeyError(f"`summary` missing columns: {missing}. Has: {list(summary.columns)}")

summary = summary[list(need_cols)].copy()
summary["neuron_id"] = summary["neuron_id"].astype(int)

# Backup original
backup_path = csv_path.replace(".csv", f".backup_{datetime.now():%Y%m%d_%H%M%S}.csv")
shutil.copy(csv_path, backup_path)

# Drop existing columns if present, then merge
df = df.drop(columns=[c for c in ["n_output_synapses", "n_downstream_neurons_ge4"] if c in df.columns])
df = df.merge(summary, on="neuron_id", how="left")

# Fill missing and set types
df["n_output_synapses"] = df["n_output_synapses"].fillna(0).astype(int)
df["n_downstream_neurons_ge4"] = df["n_downstream_neurons_ge4"].fillna(0).astype(int)

# Save in place
df.to_csv(csv_path, index=False)
print(f"✅ Updated CSV: {csv_path}\n🗂️ Backup: {backup_path}")
